# LLaMA-7B DoRA


In [ ]:
%pip install -q --no-cache-dir --upgrade-strategy only-if-needed \
  "transformers==4.44.2" \
  "accelerate==0.34.2" \
  "datasets==2.21.0" \
  "sentencepiece" \
  "safetensors" \
  "tqdm"

In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)

In [ ]:
from pathlib import Path
import gc
import json
import math
import os
import random
import re
import time
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Configuration

Point `DORA_ROOT` to the folder containing `datasets/`. In Colab, upload/copy the Desktop `DoRA` folder to `/content/DoRA` or Google Drive and edit the path below. Set `BASE_MODEL` to your accessible LLaMA-7B Hugging Face path or local checkpoint.

In [ ]:
# Path containing datasets/boolq, datasets/piqa, etc.
DORA_ROOT = Path('/content/DoRA')
if not (DORA_ROOT / 'datasets').exists():
    # Common Google Drive location if you upload the folder there.
    DORA_ROOT = Path('/content/drive/MyDrive/DoRA')

DATA_ROOT = DORA_ROOT / 'datasets'
OUTPUT_ROOT = DORA_ROOT / 'outputs_table15_reproduction'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# BASE_MODEL = 'yahma/llama-13b-hf'
# BASE_MODEL = 'meta-llama/Llama-2-7b-hf'
# BASE_MODEL = 'meta-llama/Meta-Llama-3-8B'
BASE_MODEL = "yahma/llama-7b-hf"

RUN_RANKS = [16, 32]
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'up_proj', 'down_proj']

# Paper Table 8 for LLaMA-7B DoRA: alpha=2r, dropout=0.05, AdamW, LR=2e-4, linear scheduler,
# batch size 16, warmup steps 100, epochs 3, target modules Q,K,V,Up,Down.
EPOCHS = 3
LEARNING_RATE = 2e-4
WARMUP_STEPS = 100
DROPOUT = 0.05
MAX_LENGTH = 256
MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
WEIGHT_DECAY = 0.0
MAX_GRAD_NORM = 1.0

# Leave as None for full reproduction. Use small values only for debugging notebook mechanics.
MAX_TRAIN_SAMPLES = 10000
MAX_EVAL_SAMPLES = 200

TASK_DIRS = {
    'BoolQ': 'boolq',
    'PIQA': 'piqa',
    'SIQA': 'social_i_qa',
    'HellaSwag': 'hellaswag',
    'WinoGrande': 'winogrande',
    'ARC-e': 'ARC-Easy',
    'ARC-c': 'ARC-Challenge',
    'OBQA': 'openbookqa',
}

print('DORA_ROOT:', DORA_ROOT)
print('DATA_ROOT exists:', DATA_ROOT.exists())
print('OUTPUT_ROOT:', OUTPUT_ROOT)

## Dataset loading and prompt formatting

The local dataset files are already instruction formatted. Training uses `instruction + output`. Evaluation builds the same instruction prefix and scores each allowed answer from `Answer format: ...`.

In [ ]:
def load_json_list(path: Path) -> List[dict]:
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f'{path} must contain a JSON list')
    return data


def example_prompt(example: dict) -> str:
    instruction = str(example.get('instruction', '')).strip()
    extra_input = str(example.get('input', '')).strip()
    if extra_input:
        instruction = instruction + '\n\n' + extra_input
    return '### Instruction:\n' + instruction + '\n\n### Response:\n'


def example_train_text(example: dict) -> str:
    return example_prompt(example) + str(example['output']).strip() + '</s>'


def extract_choices(instruction: str) -> List[str]:
    match = re.search(r'Answer format:\s*([^\n]+)', instruction, flags=re.IGNORECASE)
    if not match:
        raise ValueError('Could not find Answer format in instruction: ' + instruction[:200])
    raw = match.group(1).strip()
    choices = [c.strip().lower() for c in raw.split('/') if c.strip()]
    if not choices:
        raise ValueError('No choices parsed from: ' + raw)
    return choices


def canonical_answer(answer: str) -> str:
    answer = str(answer).strip().lower()
    # Handles strings such as 'the correct answer is answer1'.
    if 'the correct answer is ' in answer:
        answer = answer.split('the correct answer is ', 1)[1].strip()
    return answer.split()[0].strip('.,;:')


def load_all_train_examples() -> List[dict]:
    examples = []
    for task, dirname in TASK_DIRS.items():
        path = DATA_ROOT / dirname / 'train.json'
        task_examples = load_json_list(path)
        for ex in task_examples:
            ex = dict(ex)
            ex['task'] = task
            examples.append(ex)
        print(f'{task:10s} train: {len(task_examples):6d}')
    random.Random(SEED).shuffle(examples)
    if MAX_TRAIN_SAMPLES is not None:
        examples = examples[:MAX_TRAIN_SAMPLES]
    print('Total train examples:', len(examples))
    return examples


def load_eval_examples(task: str) -> List[dict]:
    path = DATA_ROOT / TASK_DIRS[task] / 'test.json'
    examples = load_json_list(path)
    if MAX_EVAL_SAMPLES is not None:
        examples = examples[:MAX_EVAL_SAMPLES]
    return examples

# Show schema sanity.
for task, dirname in TASK_DIRS.items():
    sample = load_json_list(DATA_ROOT / dirname / 'train.json')[0]
    print('\n', task, sample.keys())
    print(sample['instruction'][:240].replace('\n', ' '))
    print('choices:', extract_choices(sample['instruction']), 'answer:', canonical_answer(sample['answer']))

In [ ]:
class SFTDataset(Dataset):
    def __init__(self, examples: Sequence[dict], tokenizer, max_length: int):
        self.examples = list(examples)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        text = example_train_text(self.examples[idx])
        enc = self.tokenizer(text, truncation=True, max_length=self.max_length, padding=False)
        return {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask']}


def make_collator(tokenizer):
    def collate(batch):
        max_len = max(len(x['input_ids']) for x in batch)
        input_ids, attention_mask = [], []
        for item in batch:
            pad = max_len - len(item['input_ids'])
            input_ids.append(item['input_ids'] + [tokenizer.pad_token_id] * pad)
            attention_mask.append(item['attention_mask'] + [0] * pad)
        input_ids = torch.tensor(input_ids, dtype=torch.long)
        attention_mask = torch.tensor(attention_mask, dtype=torch.long)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}
    return collate

## DoRA from scratch

For a frozen pretrained weight `W0`, DoRA trains a magnitude vector `m` and low-rank direction update `BA`:

$$W' = m \frac{W_0 + BA}{\lVert W_0 + BA \rVert_c}$$

PyTorch `Linear` stores weights as `(out_features, in_features)`, so this implementation normalizes each output row. This is equivalent to column-wise normalization under the transposed convention used by `F.linear`.

In [ ]:
class DoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: int, dropout: float):
        super().__init__()
        if not isinstance(base, nn.Linear):
            raise TypeError(f'DoRALinear expects nn.Linear, got {type(base)}')
        self.in_features = base.in_features
        self.out_features = base.out_features
        self.rank = int(rank)
        self.alpha = int(alpha)
        self.scaling = float(alpha) / float(rank)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        self.weight = nn.Parameter(base.weight.detach().clone(), requires_grad=False)
        if base.bias is None:
            self.bias = None
        else:
            self.bias = nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        device = self.weight.device
        dtype = self.weight.dtype
        self.lora_A = nn.Parameter(torch.empty(rank, self.in_features, device=device, dtype=dtype))
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, rank, device=device, dtype=dtype))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        magnitude = torch.linalg.vector_norm(self.weight.float(), ord=2, dim=1).to(device=device, dtype=dtype)
        self.magnitude = nn.Parameter(magnitude)


    def merged_weight(self):
        directional = self.weight + self.scaling * (self.lora_B @ self.lora_A)
        norm = torch.linalg.vector_norm(directional.float(), ord=2, dim=1, keepdim=True).to(directional.dtype).clamp_min(1e-6)
        return self.magnitude[:, None] * directional / norm

    def forward(self, x):
        return F.linear(self.dropout(x), self.merged_weight(), self.bias)


def replace_target_linears_with_dora(model: nn.Module, target_modules: Iterable[str], rank: int, alpha: int, dropout: float) -> int:
    modules = dict(model.named_modules())
    replacements = []
    target_modules = tuple(target_modules)
    for name, module in modules.items():
        if isinstance(module, nn.Linear) and any(target == name.split('.')[-1] or target in name for target in target_modules):
            parent_name, child_name = name.rsplit('.', 1) if '.' in name else ('', name)
            parent = modules[parent_name] if parent_name else model
            replacements.append((parent, child_name, name))
    for parent, child_name, full_name in replacements:
        old = getattr(parent, child_name)
        setattr(parent, child_name, DoRALinear(old, rank=rank, alpha=alpha, dropout=dropout))
    return len(replacements)


def freeze_all_then_unfreeze_dora(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False
    for module in model.modules():
        if isinstance(module, DoRALinear):
            module.lora_A.requires_grad = True
            module.lora_B.requires_grad = True
            module.magnitude.requires_grad = True


def trainable_parameter_summary(model: nn.Module) -> Tuple[int, int, float]:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total, 100.0 * trainable / total

# Tiny startup identity test.
_test = nn.Linear(7, 4, bias=False)
_wrap = DoRALinear(_test, rank=2, alpha=4, dropout=0.0)
_x = torch.randn(3, 7)
with torch.no_grad():
    assert torch.allclose(_test(_x), _wrap(_x), atol=1e-5)
print('DoRALinear startup identity test passed.')

## Model loading

This loads LLaMA-7B in fp16/bf16 and then replaces target modules with DoRA. The base weights stay frozen, so only DoRA `A`, `B`, and `magnitude` train.

In [ ]:
def load_tokenizer(base_model: str):
    tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'right'
    return tokenizer


def load_dora_model(base_model: str, rank: int):
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModelForCausalLM.from_pretrained(
        base_model,
        torch_dtype=dtype,
        device_map='auto',
        low_cpu_mem_usage=True,
    )
    if hasattr(model, 'config'):
        model.config.use_cache = False
    model.gradient_checkpointing_enable()
    if hasattr(model, 'enable_input_require_grads'):
        model.enable_input_require_grads()

    alpha = 2 * rank
    replaced = replace_target_linears_with_dora(model, TARGET_MODULES, rank=rank, alpha=alpha, dropout=DROPOUT)
    freeze_all_then_unfreeze_dora(model)
    trainable, total, pct = trainable_parameter_summary(model)
    print(f'Rank {rank}: replaced {replaced} linear layers')
    print(f'Rank {rank}: trainable {trainable:,} / {total:,} = {pct:.2f}%')
    return model

## Training loop

This is a plain PyTorch loop, not PEFT/DoRA library code. It matches the paper hyperparameters as closely as possible for LLaMA-7B Table 8: AdamW, LR `2e-4`, linear schedule, warmup `100`, epochs `3`, dropout `0.05`, and effective batch size `16`.

In [ ]:
def save_dora_adapter(model: nn.Module, path: Path, rank: int):
    path.parent.mkdir(parents=True, exist_ok=True)
    state = {}
    for name, module in model.named_modules():
        if isinstance(module, DoRALinear):
            state[name + '.lora_A'] = module.lora_A.detach().cpu()
            state[name + '.lora_B'] = module.lora_B.detach().cpu()
            state[name + '.magnitude'] = module.magnitude.detach().cpu()
    torch.save({'rank': rank, 'alpha': 2 * rank, 'target_modules': TARGET_MODULES, 'state_dict': state}, path)
    print('Saved adapter:', path)


def train_one_rank(rank: int, tokenizer) -> nn.Module:
    train_examples = load_all_train_examples()
    dataset = SFTDataset(train_examples, tokenizer, MAX_LENGTH)
    loader = DataLoader(dataset, batch_size=MICRO_BATCH_SIZE, shuffle=True, collate_fn=make_collator(tokenizer), num_workers=2, pin_memory=True)

    model = load_dora_model(BASE_MODEL, rank)
    model.train()

    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    update_steps_per_epoch = math.ceil(len(loader) / GRADIENT_ACCUMULATION_STEPS)
    total_update_steps = EPOCHS * update_steps_per_epoch
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_update_steps)

    global_step = 0
    running = 0.0
    start = time.time()
    optimizer.zero_grad(set_to_none=True)

    for epoch in range(EPOCHS):
        pbar = tqdm(loader, desc=f'rank {rank} epoch {epoch + 1}/{EPOCHS}')
        for step, batch in enumerate(pbar):
            batch = {k: v.to(model.device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss / GRADIENT_ACCUMULATION_STEPS
            loss.backward()
            running += float(loss.detach().cpu())

            if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or (step + 1) == len(loader):
                torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), MAX_GRAD_NORM)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
                if global_step % 10 == 0:
                    pbar.set_postfix({'loss': running / 10, 'lr': scheduler.get_last_lr()[0]})
                    running = 0.0

    print(f'Rank {rank} training finished in {(time.time() - start) / 60:.1f} min, updates={global_step}')
    save_dora_adapter(model, OUTPUT_ROOT / f'dora_llama7b_rank{rank}.pt', rank)
    return model

## Multiple-choice evaluation

For every candidate in the example's `Answer format`, this evaluates the conditional negative log likelihood of the completion `the correct answer is <candidate>`. The selected answer is the candidate with the lowest completion loss.

In [ ]:
def candidate_nll(model, tokenizer, prompt: str, completion: str, max_length: int) -> float:
    prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
    full = prompt + completion
    enc = tokenizer(full, truncation=True, max_length=max_length, return_tensors='pt', add_special_tokens=False)
    input_ids = enc.input_ids.to(model.device)
    attention_mask = enc.attention_mask.to(model.device)
    labels = input_ids.clone()
    # Mask prompt tokens. If truncation removed part of the prompt, this still masks what remains up to prompt length.
    mask_len = min(len(prompt_ids), labels.shape[1])
    labels[:, :mask_len] = -100
    with torch.no_grad():
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    return float(out.loss.detach().cpu())


def predict_example(model, tokenizer, example: dict) -> str:
    prompt = example_prompt(example)
    choices = extract_choices(example['instruction'])
    scores = []
    for choice in choices:
        completion = 'the correct answer is ' + choice
        scores.append(candidate_nll(model, tokenizer, prompt, completion, MAX_LENGTH))
    return choices[int(np.argmin(scores))]


def evaluate_task(model, tokenizer, task: str) -> Dict[str, object]:
    examples = load_eval_examples(task)
    correct = 0
    rows = []
    model.eval()
    for ex in tqdm(examples, desc=f'eval {task}'):
        pred = predict_example(model, tokenizer, ex)
        gold = canonical_answer(ex['answer'])
        ok = pred == gold
        correct += int(ok)
        rows.append({'task': task, 'prediction': pred, 'answer': gold, 'correct': ok, 'instruction': ex['instruction']})
    acc = 100.0 * correct / len(examples)
    return {'task': task, 'accuracy': acc, 'n': len(examples), 'rows': rows}


def evaluate_all_tasks(model, tokenizer, rank: int) -> Dict[str, float]:
    accuracies = {}
    all_rows = []
    for task in TASK_DIRS:
        result = evaluate_task(model, tokenizer, task)
        accuracies[task] = result['accuracy']
        all_rows.extend(result['rows'])
        print(f"rank {rank} {task}: {result['accuracy']:.1f} over {result['n']} examples")
    pd.DataFrame(all_rows).to_csv(OUTPUT_ROOT / f'predictions_rank{rank}.csv', index=False)
    return accuracies

## Run ranks 16 and 32

This is the main reproduction cell. It will take a long time on full data. For a quick pipeline check, set `MAX_TRAIN_SAMPLES = 128`, `MAX_EVAL_SAMPLES = 32`, and `RUN_RANKS = [16]`, then restart runtime and run all cells.

In [ ]:
def table_row_from_accuracies(rank: int, accuracies: Dict[str, float], trainable_pct: Optional[float] = None) -> Dict[str, object]:
    row = {'Method': 'DoRA (Ours)', 'rank r': rank}
    paper_pct = {16: 0.43, 32: 0.84, 64: 1.65, 8: 0.22, 4: 0.11}.get(rank, trainable_pct)
    row['# Params (%)'] = paper_pct
    for task in TASK_DIRS:
        row[task] = round(float(accuracies[task]), 1)
    row['Avg.'] = round(float(np.mean([accuracies[t] for t in TASK_DIRS])), 1)
    return row


tokenizer = load_tokenizer(BASE_MODEL)
rows = []

for rank in RUN_RANKS:
    print('\n' + '=' * 80)
    print('Training/evaluating rank', rank)
    model = train_one_rank(rank, tokenizer)
    acc = evaluate_all_tasks(model, tokenizer, rank)
    rows.append(table_row_from_accuracies(rank, acc))
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results = pd.DataFrame(rows)
out_path = OUTPUT_ROOT / 'table15_llama7b_dora_reproduced.csv'
results.to_csv(out_path, index=False)
print('Saved:', out_path)
results